# Install and Import Required Libs

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
!pip install -q langchain_text_splitters
!pip install -q langchain langchain-community langchain-core langchain-huggingface
!pip install -q faiss-gpu

import langchain
import langchain_core
import langchain_huggingface
import langchain_community

print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_community:", langchain_community.__version__)

/kaggle/input/datasets/kalyandath/science-dataset/physics_pages.csv
/kaggle/input/wiki-dataset/Wiki_knowledge_raw_gem.csv
/kaggle/input/wiki-dataset/Wiki_knowledge_raw.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
langchain: 1.2.15
langchain_core: 1.4.9
langchain_community: 0.4.2


# Set Required Inputs

In [3]:
DATA_PATH = "/kaggle/input/datasets/kalyandath/science-dataset/physics_pages.csv"
WIKI_DATH_PATH = "/kaggle/input/wiki-dataset/Wiki_knowledge_raw_gem.csv"
data = pd.read_csv(DATA_PATH)
wiki_data = pd.read_csv(WIKI_DATH_PATH)

# Inspect the dataframes

## Show first few data for dataset data.

In [4]:
data.head()

,page,section_title,text
0,Electric field,Description,The electric field is defined at each point in...
1,Electric field,Mathematical formulation,Electric fields are caused by electric charges...
2,Electric field,Electrostatic fields,Electrostatic fields are electric fields that ...
3,Electric field,Electrodynamic fields,Electrodynamic fields are electric fields whic...
4,Electric field,Energy in the electric field,The total energy per unit volume stored by the...


## Show first few data for wikipedia data.

In [5]:
wiki_data.head()

,title,content
0,API gravity,"The American Petroleum Institute gravity, or A..."
1,Absorption (electromagnetic radiation),"In physics, absorption of electromagnetic radi..."
2,Absorption spectroscopy,Absorption spectroscopy is spectroscopy that i...
3,Active transport,"In cellular biology, active transport is the m..."
4,Aerodynamics,Aerodynamics (from Ancient Greek ἀήρ (aḗr) 'a...


## More info on dataset data

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   page           4096 non-null   object
 1   section_title  4096 non-null   object
 2   text           4096 non-null   object
dtypes: object(3)
memory usage: 96.1+ KB


## More info on Wikipedia data.

In [7]:
wiki_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    612 non-null    object
 1   content  612 non-null    object
dtypes: object(2)
memory usage: 9.7+ KB


# Apply Normalization to clean the data.

## Remove any empty rows from the dataset data.

In [8]:
empty_content_count = data["text"].isna().sum()
print("Field with no data: ", empty_content_count)
print("Since no empty data, no need to remove any rows")

Field with no data:  0
Since no empty data, no need to remove any rows


## Remove any empty rows from the wikipedia data.

In [9]:
empty_content_count = wiki_data["content"].isna().sum()
print("Field with no data: ", empty_content_count)
print("Since no empty data, no need to remove any rows")

Field with no data:  0
Since no empty data, no need to remove any rows


## Remove content having less than 50 char length.

In [10]:
print(f"Content with less than 50 chars:{(data["text"].str.len() < 50).sum()}")
print(f"Content with less than 50 chars for wiki data:{(wiki_data["content"].str.len() < 50).sum()}")
print("Remove contents having less than 50 characters.")

25
Remove contents having less than 50 characters.


In [12]:
data = data.loc[data["text"].str.len() > 50]

# Function to clean the data content.

In [ ]:
import re

def clean_text(text):

    # 1. Remove the wikipedia citation markers
    # [1], [23]
    text = re.sub(r'\[\d+\]\s*', "", text)

    # 2. Remove the section headers
    # === History ===, === References ===
    text = re.sub(r'^={2,}.*?={2,}', "", text, flags=re.MULTILINE)

    # 3. Remove "Source:" labels
    text = re.sub(r"Source:\s*(?=$|\n)","", text)

    # 4. Remove common latex commands
    # \frac, \alpha, \displaystyle
    text = re.sub(r'\\[a-zA-Z]+', " ", text)

    # 5. Remove leftover latex symbols
    text = re.sub(r"[{}\\^~]", " ", text)

    # 6. Remove page-number styes
    # : 327 , :  469-470
    text = re.sub(r":\s*\d+(?:-\d+)?\b", "", text)

    # 7 . Normalize whitespaces
    text = "\n".join(line.strip() for line in text.splitlines())

    # 8. Collapse multiple blank lines
    text = re.sub(r"\n{2,}", "\n", text)

    # 9. Convert everything into single para.
    text = text.replace("\n"," ").replace("\r", " ")

    # 10. Collapse sequence of whtiespace into a single space
    text = re.sub(r"\s+", " ",text)


    return text.strip()

# Process and clean content, create document for chunking.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Define the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 300,
    separators = [". ", "? ", "! ", ", ", " ", ""]
)
print("Starting to create documents...")
documents = []

for index, row in data.iterrows():
    page  = str(row["page"])
    title = str(row["section_title"])
    raw_text  = str(row["text"])
    text = clean_text(raw_text)
    context_string = f"{page}:{title}\n\n{text}"

    documents.append(Document(
        page_content = context_string,
        metadata = {"source": title}
    ))

print("Successfully processed documents from science dataset:", len(documents))

for index, row in wiki_data.iterrows():
    title = str(row['title'])
    raw_text = str(row["content"])

    text = clean_text(raw_text)
    context_string = f"{title}:\n\n{text}"

    documents.append(Document(
        page_content = context_string,
        metadata = {"source": title}
    ))
print("Successfully processed documents from wiki dataset. Total Documents:", len(documents))

Starting to create documents...
Successfully processed documents from science dataset: 4070
Successfully processed documents from wiki dataset. Total Documents: 4682


# Chunking documents.

In [15]:
filtered_documents = [doc for doc in documents if len(doc.page_content) > 100]
chunks = text_splitter.split_documents(filtered_documents)
print(f"Splitted douments into {len(chunks)} chunks....")

Splitted douments into 31338 chunks....


# Create faiss index with the chunks.

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {'device':'cuda'}
)

vector_store = FAISS.from_documents(
    documents  = chunks,
    embedding = embeddings,
    distance_strategy = DistanceStrategy.COSINE
)

vector_store.save_local("/kaggle/working/faiss_index")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Look into the FAISS index.

In [17]:
faiss_index = vector_store.index

total_vectors = faiss_index.ntotal
print(f"Total vectors in the index: {total_vectors}") # equal to the number of chunks

embedding_matrix = faiss_index.reconstruct_n(0, total_vectors)
embedding_matrix

Total vectors in the index: 31338


array([[ 0.00997513,  0.06422407, -0.02746625, ...,  0.04903989,
        -0.01690243, -0.00686218],
       [-0.02857855,  0.04561146, -0.02898386, ...,  0.06238057,
        -0.00869794,  0.03057728],
       [-0.04699242,  0.01989065, -0.0556299 , ...,  0.08089621,
        -0.00098497,  0.02705218],
       ...,
       [-0.09065121, -0.01518891, -0.01014029, ...,  0.05976262,
         0.02544466,  0.02168891],
       [-0.08007357,  0.0384741 ,  0.01903544, ...,  0.06264371,
         0.02515977,  0.04163513],
       [-0.1013193 ,  0.029567  ,  0.0031213 , ...,  0.02329884,
        -0.04080019,  0.10667023]], dtype=float32)

# Example query to the rag for testing.

In [19]:
query = "What is emissivity?"

docs_scores = vector_store.similarity_search_with_score(query, k=3)
top_doc = docs_scores[0][0]
print(top_doc.page_content)
print("-"*20)
for doc,score in docs_scores:
    print(f"Score(distance): {score:.4f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}....\n")


Emissivity:Mathematical definitions

In its most general form, emissivity can specified for a particular wavelength, direction, and polarization. However, the form of emissivity that most commonly used is the hemispherical total emissivity, which considers emissions as totaled over all wavelengths, directions, and polarizations, given a particular temperature.: 60 Some specific forms of emissivity are detailed below.
Score(distance): 0.4633
Source: Mathematical definitions
Content: Emissivity:Mathematical definitions

In its most general form, emissivity can specified for a particular wavelength, direction, and polarization. However, the form of emissivity that most commonly used is the hemispherical total emissivity, which considers emissions as totaled over all wavelengths, directions, and polarizations, given a particular temperature.: 60 Some specific forms of emissivity are detailed below.....

Score(distance): 0.6151
Source: Summary
Content: Emissivity:Summary

The emissivity of 

# Export FAISS Index to ZIP

In [20]:
import shutil
shutil.make_archive("/kaggle/working/faiss_index_download", "zip", "/kaggle/working/faiss_index")

'/kaggle/working/faiss_index_download.zip'